# Phase 2 - Step 9: Model Versioning

This notebook persists the winning trained production pipeline, writes actual metadata to `models/v1/metadata.json`, and records optional MLflow tracking.

In [1]:
import pandas as pd
import numpy as np
import json
import joblib
import datetime
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from xgboost import XGBClassifier
from sklearn.metrics import precision_score, recall_score, f1_score, roc_auc_score

models_dir = Path("models/v1")
models_dir.mkdir(parents=True, exist_ok=True)
processed_dir = Path("data/processed")

df = pd.read_csv(processed_dir / "employee_attrition_processed.csv")

# Feature engineering
df['income_per_year_at_company'] = df['MonthlySalary'] * 12.0 / (df['YearsAtCompany'] + 1.0)
df['promotion_gap_ratio'] = (2026.0 - df['LastPromotionYear']) / (df['YearsAtCompany'] + 1.0)
df['overtime_ratio'] = df['OvertimeHoursPerMonth'] / 160.0
df['leave_utilization'] = df['LeavesTaken'] / 20.0
df['work_life_satisfaction'] = df['WorkLifeBalanceScore'] * df['CustomerSatisfaction']

target_col = 'AttritionRisk'
y = df[target_col].map({'Yes': 1, 'No': 0}).values
drop_cols = ['EmployeeID', 'Name', 'PhoneNumber', 'JoiningDate', 'LastLeaveDate', target_col, 'CountryCode']
X = df.drop(columns=[c for c in drop_cols if c in df.columns])

num_cols = X.select_dtypes(include=['int64', 'float64']).columns.tolist()
cat_cols = X.select_dtypes(include=['object', 'string']).columns.tolist()

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

preprocessor = ColumnTransformer(
    transformers=[
        ('num', Pipeline([('imputer', SimpleImputer(strategy='median')), ('scaler', StandardScaler())]), num_cols),
        ('cat', Pipeline([('imputer', SimpleImputer(strategy='most_frequent')), ('ohe', OneHotEncoder(handle_unknown='ignore', sparse_output=False))]), cat_cols)
    ]
)

pos_weight = (len(y_train) - sum(y_train)) / sum(y_train)
xgb_clf = XGBClassifier(scale_pos_weight=pos_weight, eval_metric='logloss', random_state=42)

full_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', xgb_clf)
])

full_pipeline.fit(X_train, y_train)

# Evaluate on test split
y_pred = full_pipeline.predict(X_test)
y_proba = full_pipeline.predict_proba(X_test)[:, 1]

precision_val = float(precision_score(y_test, y_pred))
recall_val = float(recall_score(y_test, y_pred))
f1_val = float(f1_score(y_test, y_pred))
roc_auc_val = float(roc_auc_score(y_test, y_proba))

# Save pipeline artifact
pipeline_path = models_dir / "attrition_pipeline.joblib"
joblib.dump(full_pipeline, pipeline_path)
print(f"Saved pipeline to {pipeline_path}")

# Write real metadata
metadata = {
    "model_name": "attrition_risk_classifier",
    "version": "v1",
    "algorithm": "XGBoost",
    "training_date": datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
    "dataset_name": "employee_attrition_processed.csv",
    "target_column": target_col,
    "positive_class": "Yes (1)",
    "negative_class": "No (0)",
    "train_samples": int(len(X_train)),
    "test_samples": int(len(X_test)),
    "feature_count": int(len(num_cols) + len(cat_cols)),
    "numerical_features": num_cols,
    "categorical_features": cat_cols,
    "precision": round(precision_val, 4),
    "recall": round(recall_val, 4),
    "f1_score": round(f1_val, 4),
    "roc_auc": round(roc_auc_val, 4),
    "test_size": 0.2,
    "random_state": 42
}

meta_path = models_dir / "metadata.json"
with open(meta_path, "w", encoding="utf-8") as f:
    json.dump(metadata, f, indent=4)
print(f"Saved metadata to {meta_path}")
print(json.dumps(metadata, indent=2))


Saved pipeline to models\v1\attrition_pipeline.joblib
Saved metadata to models\v1\metadata.json
{
  "model_name": "attrition_risk_classifier",
  "version": "v1",
  "algorithm": "XGBoost",
  "training_date": "2026-08-29 17:25:25",
  "dataset_name": "employee_attrition_processed.csv",
  "target_column": "AttritionRisk",
  "positive_class": "Yes (1)",
  "negative_class": "No (0)",
  "train_samples": 400,
  "test_samples": 100,
  "feature_count": 23,
  "numerical_features": [
    "Age",
    "EducationLevel",
    "MonthlySalary",
    "OvertimeHoursPerMonth",
    "LeavesTaken",
    "ProjectsHandled",
    "TrainingHours",
    "CustomerSatisfaction",
    "LastPromotionYear",
    "YearsAtCompany",
    "WorkLifeBalanceScore",
    "PerformanceRating",
    "CustomerSatisfaction_missing",
    "income_per_year_at_company",
    "promotion_gap_ratio",
    "overtime_ratio",
    "leave_utilization",
    "work_life_satisfaction"
  ],
  "categorical_features": [
    "Gender",
    "Department",
    "JobRol

In [2]:
# Optional MLflow Tracking
try:
    import mlflow
    import mlflow.sklearn
    
    mlflow.set_experiment("enterprise_hr_attrition")
    with mlflow.start_run(run_name="xgboost_v1"):
        mlflow.log_params({
            "model_version": "v1",
            "algorithm": "XGBoost",
            "test_size": 0.2,
            "random_state": 42
        })
        mlflow.log_metrics({
            "precision": precision_val,
            "recall": recall_val,
            "f1_score": f1_val,
            "roc_auc": roc_auc_val
        })
        mlflow.sklearn.log_model(full_pipeline, "model")
        print("Successfully logged run to MLflow experiment 'enterprise_hr_attrition'.")
except Exception as e:
    print(f"MLflow optional logging skipped (non-blocking): {e}")


MLflow optional logging skipped (non-blocking): No module named 'mlflow'
